In [4]:
! pip install numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 21.1 MB/s eta 0:00:0000:0100:01


In [7]:
import sys
import os
import glob
import numpy as np
import time
from scipy.spatial import cKDTree

In [8]:
class FoxTree:
    def __init__(self, points_array, radius, vertical_resolution, min_pts_per_cluster):
        """
        :param points_array: numpy array of shape (N, 3) containing x, y, z
        """
        self.radius = radius
        self.vertical_resolution = vertical_resolution
        self.min_pts_seeds = min_pts_per_cluster
        
        # Store original data
        self.points_data = points_array
        self.num_pts = len(points_array)
        
        # Initialize attributes to track state
        # tree_ids: -1 means unassigned
        self.tree_ids = np.full(self.num_pts, -1, dtype=int)
        
        # Bounding box
        self.z_min = np.min(points_array[:, 2])
        self.z_max = np.max(points_array[:, 2])
        
        # Tracking assigned points
        self.parsed_pt_indices = [] # List of indices
        self.next_tree_id = 0
        
        # Map of tree_id -> list of point indices (for final output)
        self.trees = {}

    def generate_tree_clusters(self, pt_clusters):
        for cluster_indices in pt_clusters:
            current_id = self.next_tree_id
            self.next_tree_id += 1
            self.trees[current_id] = []
            for idx in cluster_indices:
                self.tree_ids[idx] = current_id
                self.trees[current_id].append(idx)

